# [Deepmath] 10.2. Analyse de texte - tensorflow - IMDB

In [2]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

## Partie A. Données

In [3]:
from tensorflow.keras.datasets import imdb

In [4]:
nb_mots_total = 1000   # On ne garde que les n=1000 mots les plus fréquents 
(X_train_data, Y_train), (X_test_data, Y_test) = imdb.load_data(num_words=nb_mots_total)

17464789/17464789 [==============================] - 15s 1us/step


In [64]:
#X_train_data[454]

## Partie A bis. Afficher d'un texte

In [35]:
# Pour comprendre la structure des données
index_mots = imdb.get_word_index()
index_mots_inverse = dict([(value, key) for key, value in index_mots.items()])
critique = " ".join(index_mots_inverse.get(i - 3, '??') for i in X_train_data[1])
index_mots_inverse[2]

'and'

- The .get(index, replace_value) helps extract a value from a dictionnary corresponding to the index and in case the value is unavailable, the replace_value is given.

-  L'objet "index_mots_inverse" est comme un dictionnaire des mots

- L'objet "X_train_data[num]" est un vecteur des nombres représentant un commentaire. Chaque nombre correspond représente un mot et indique la position du mot en question dans la liste des 1000 mots anglais les plus fréquents.


In [5]:
# Afficher une critique et sa note
def affiche_texte(num):
    index_mots = imdb.get_word_index()
    index_mots_inverse = dict([(value, key) for (key, value) in index_mots.items()])
    critique_mots = ' '.join([index_mots_inverse.get(i - 3, '??') for i in X_train_data[num]])
    print("Critique :\n", critique_mots)
    print("Note 0 (négatif) ou 1 (positif) ? :", Y_train[num])
    print("Critique (sous forme brute) :\n", X_train_data[num])
    return

affiche_texte(123)   # affichage de la critique numéro 123

1641221/1641221 [==============================] - 1s 1us/step
Critique :
 ?? beautiful and ?? movie ?? ?? great ?? good acting and one of the most ?? movies i have seen in a while i never saw such an interesting setting when i was in ?? my wife liked it so much she ?? me to ?? on and rate it so other would enjoy too
Note 0 (négatif) ou 1 (positif) ? : 1
Critique (sous forme brute) :
 [1, 307, 5, 2, 20, 2, 2, 87, 2, 52, 116, 5, 31, 7, 4, 91, 2, 102, 13, 28, 110, 11, 6, 137, 13, 115, 219, 141, 35, 221, 956, 54, 13, 16, 11, 2, 61, 322, 423, 12, 38, 76, 59, 2, 72, 8, 2, 23, 5, 967, 12, 38, 85, 62, 358, 99]


## Partie A ter. Données sous forme de vecteurs

In [41]:
len(X_train_data[0])

218

In [6]:
def vectorisation_critiques(X_data):
    vecteurs = np.zeros((len(X_data), nb_mots_total)) # 25000 lines X 1000 colonnes
    for i in range(len(X_data)): # Pour chaque ligne
        for c in X_data[i]: # On place à la cième position du vecteur le chiffre 1.0 avec c les 
            vecteurs[i,c] = 1.0
    return vecteurs

X_train = vectorisation_critiques(X_train_data)
X_test = vectorisation_critiques(X_test_data)

## Partie B. Réseau 

- "loss" représente la fonction coût c'est à dire la fonction d'erreur

- "binary_crossentropy" est une fonction d'erreur qui convient pour une classification binaire

- "categorical_crossentropy" par contre convient pout une classification en plusieurs catégories (comme pour la reconnaissance des chiffres de 0 à 9)

In [7]:
modele = Sequential()
p = 5
modele.add(Dense(p, input_dim=nb_mots_total, activation='relu'))
modele.add(Dense(p, activation='relu'))
modele.add(Dense(p, activation='relu'))
modele.add(Dense(1, activation='sigmoid'))
modele.compile(loss='binary_crossentropy', optimizer='sgd', metrics=['accuracy'])

In [59]:
dir(keras.optimizers)

['Adadelta',
 'Adagrad',
 'Adam',
 'Adamax',
 'Ftrl',
 'Nadam',
 'Optimizer',
 'RMSprop',
 'SGD',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_sys',
 'deserialize',
 'experimental',
 'get',
 'legacy',
 'schedules',
 'serialize']

## Partie C. Apprentissage

In [55]:
modele.fit(X_train, Y_train, epochs=5, batch_size=32)

Epoch 1/5
782/782 [==============================] - 39s 49ms/step - loss: 0.2432 - accuracy: 0.8974
Epoch 2/5
782/782 [==============================] - 48s 61ms/step - loss: 0.2381 - accuracy: 0.8984
Epoch 3/5
782/782 [==============================] - 39s 50ms/step - loss: 0.2347 - accuracy: 0.9007
Epoch 4/5
782/782 [==============================] - 39s 50ms/step - loss: 0.2320 - accuracy: 0.9004
Epoch 5/5
782/782 [==============================] - 40s 51ms/step - loss: 0.2294 - accuracy: 0.9028


### Les poids : coefficients et biais

In [66]:
# Coefficients pour la couche 1
# Le ième vecteur ligne repésente les coefficients de la ième variable d'entrée
modele.get_weights()[0]

array([[ 0.04928654,  0.05253553,  0.0368171 ,  0.03135411, -0.00619902],
       [ 0.39337108, -0.057094  ,  0.02473009,  0.13515957,  0.07301307],
       [ 0.3471264 ,  0.04006891,  0.00993855,  0.20869292,  0.02139232],
       ...,
       [-0.06282019,  0.01132836, -0.02003509,  0.04018082,  0.0080696 ],
       [-0.02566114,  0.10072031, -0.05176366, -0.0582711 ,  0.0288452 ],
       [-0.1537814 ,  0.01434813, -0.0036105 , -0.04527723,  0.02959951]],
      dtype=float32)

In [67]:
# Biais pour la couche 1
# Le ième élément du vecteur correspond au biais du ième neurone de la couche
modele.get_weights()[1]

array([0.35912052, 0.00914152, 0.0614307 , 0.14698996, 0.04022704],
      dtype=float32)

## Partie D. Résultats

In [38]:
X_test.shape

(25000, 1000)

In [60]:
Y_predict = modele.predict(X_test)

782/782 [==============================] - 4s 6ms/step


## Afficher une critique et sa note 

In [61]:
def affiche_texte_test(num):
    index_mots = imdb.get_word_index()
    index_mots_inverse = dict([(value, key) for (key, value) in index_mots.items()])
    critique_mots = ' '.join([index_mots_inverse.get(i - 3, '??') for i in X_test_data[num]])
    print("Critique :\n", critique_mots)
    print("Note attendue 0 (négatif) ou 1 (positif) ? :", Y_test[num])
    print("Note prédite 0 (négatif) ou 1 (positif) ? :", Y_predict[num][0])
    return

affiche_texte_test(111)   # prédiction pour la critique test numéro 111

Critique :
 ?? i at first thought this little fantasy ?? would be a little entertaining i was wrong br br a good cast ?? ?? as the ?? didn't help it any the story had every ?? possible worst case ?? that could take place in a ?? ?? ?? and none of it could possibly happen br br true the ?? of the ?? could only be ?? with the ?? help of a ?? in the ?? ?? ?? air ?? one but everything they ?? ?? the ?? and the ?? of our country if were to fall into ?? hands is ?? to the ?? ?? seriously not even the ?? can ?? over ?? our ?? ?? the case is only used to ?? ?? in this situation our ?? would have completely ?? the ?? and the whole thing would go ?? the ?? of ?? couldn't happen there would not have been a ?? ?? because the ?? ?? would have been ?? not to ?? ?? a ?? there are just too many ?? ?? to ?? such a thing from ?? br br true film's like ?? ?? and ?? gave some ?? to the ?? of us ?? ?? of the ?? but this film goes too far and fails to ?? my ?? of the ?? and that makes the experience a waste